In [1]:
# 01 · 실행 환경
from pathlib import Path
import os
Path("data").mkdir(exist_ok=True)
Path("output").mkdir(exist_ok=True)
if not Path("data/haccp_인증업체_raw.csv").exists() and not os.getenv("HACCP_API_KEY"):
    raise RuntimeError("HACCP_API_KEY 설정 또는 data/haccp_인증업체_raw.csv 필요")


In [2]:
# 02 · HACCP 인증업체 수집·전처리
import os
import time
import requests
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
SERVICE_KEY = os.environ.get('HACCP_API_KEY', '여기에_공공데이터포털_인증키')
BASE_URL = 'https://apis.data.go.kr/B553748/CertCompanyListService2/getCertCompanyListService2'
PAGE_SIZE = 100
BASE_DATE = pd.Timestamp('2026-09-08')
RAW_PATH = 'data/haccp_인증업체_raw.csv'
pd.set_option('display.width', 200)
pd.set_option('display.unicode.east_asian_width', True)

def fetch_page(session, page, retries=4):
    url = f'{BASE_URL}?serviceKey={SERVICE_KEY}&pageNo={page}&numOfRows={PAGE_SIZE}&returnType=json'
    for _ in range(retries):
        try:
            body = session.get(url, timeout=60).json().get('body')
            if body:
                return [row['item'] for row in body.get('items') or []]
        except Exception:
            pass
        time.sleep(1.5)
    print(f'  [경고] page {page} 수집 실패')
    return []

def collect_all(use_cache=True):
    if use_cache and os.path.exists(RAW_PATH):
        print(f'[수집] 캐시 사용: {RAW_PATH}')
        return pd.read_csv(RAW_PATH, dtype=str)
    session = requests.Session()
    total = int(session.get(f'{BASE_URL}?serviceKey={SERVICE_KEY}&pageNo=1&numOfRows=1&returnType=json', timeout=30).json()['body']['totalCount'])
    pages = range(1, total // PAGE_SIZE + 2)
    print(f'[수집] 전체 {total:,}건 / {len(pages)}페이지')
    with ThreadPoolExecutor(6) as ex:
        chunks = list(ex.map(lambda p: fetch_page(session, p), pages))
    df = pd.DataFrame([r for c in chunks for r in c]).drop_duplicates()
    os.makedirs('data', exist_ok=True)
    df.to_csv(RAW_PATH, index=False, encoding='utf-8-sig')
    print(f'[수집] 완료: {df.shape[0]:,}행 × {df.shape[1]}열 → {RAW_PATH}')
    return df

def preprocess(df):
    df = df.copy()
    df['인증일'] = pd.to_datetime(df['issuedate'], errors='coerce')
    df['인증종료일'] = pd.to_datetime(df['issueenddate'], errors='coerce')
    df['무기한여부'] = df['issueenddate'] == '2500-12-31'
    df['유지기간_년'] = (df['인증종료일'] - df['인증일']).dt.days / 365.25
    df.loc[df['무기한여부'], '유지기간_년'] = np.nan
    df['갱신주기수'] = (df['유지기간_년'] / 3).round()
    df['갱신횟수'] = (df['갱신주기수'] - 1).clip(lower=0)
    df['인증경과_년'] = (BASE_DATE - df['인증일']).dt.days / 365.25
    df['만료까지_일'] = (df['인증종료일'] - BASE_DATE).dt.days
    df['최초인증연도'] = df['인증일'].dt.year
    df = df.rename(columns={'appointno': '인증번호', 'company': '업체명', 'productGb': '식품구분', 'companykindNm': '업종', 'businesstypeNm': '업태', 'businessitemNm': '품목', 'area1': '시도', 'area2': '시군구'})
    return df

def eda(df):

    def head(title):
        print('\n' + '=' * 72)
        print(f' {title}')
        print('=' * 72)
    head('1. 기본 구조')
    print(f'행×열: {df.shape[0]:,} × {df.shape[1]}')
    print(f"고유 인증번호: {df['인증번호'].nunique():,}건")
    print(f"고유 업체명  : {df['업체명'].nunique():,}개")
    print(f"업체당 평균 인증 수: {len(df) / df['업체명'].nunique():.2f}건")
    head('2. 결측치')
    miss = df[['인증번호', '업체명', '시도', '시군구', '인증일', '인증종료일']].isna().sum()
    print(miss[miss >= 0].to_string())
    head('3. ★ 타당성 검증 — 이탈(만료) 업체가 데이터에 존재하는가')
    expired = (df['인증종료일'] < BASE_DATE).sum()
    print(f'인증종료일 < 기준일({BASE_DATE.date()}) : {expired:,}건')
    print(f"인증종료일 >= 기준일                  : {(df['인증종료일'] >= BASE_DATE).sum():,}건")
    print(f"인증종료일 최솟값                     : {df['인증종료일'].min().date()}")
    print("→ 만료 건이 0이면 본 API는 '현재 유효 인증'만 담은 스냅샷이며,")
    print('  이탈(포기) 업체를 직접 관측할 수 없다.')
    head('4. ★ 갱신주기 복원 가설 검증')
    v = df.dropna(subset=['갱신주기수'])
    print('갱신주기수별 실제 유지기간(년) — 평균이 3의 배수에 붙고 표준편차가 작으면 가설 성립')
    print(v.groupby('갱신주기수')['유지기간_년'].agg(건수='size', 평균='mean', 표준편차='std').round(3).to_string())
    head('5. 갱신횟수 분포')
    print(v['갱신횟수'].value_counts().sort_index().to_string())
    print(f"\n평균 갱신횟수: {v['갱신횟수'].mean():.2f}회")
    head('6. 최초인증 연도별 현재 잔존 건수 (코호트 생존)')
    print(df.groupby('최초인증연도').size().loc[2005:].to_string())
    print('→ 연도별 신규 인증 건수(인증원 통계연보)와 대조하면 코호트 생존율 산출 가능')
    head('7. 식품/축산물 구분')
    print(df['식품구분'].value_counts().to_string())
    head('8. 업종별 분포 및 평균 갱신횟수')
    print(df.groupby('업종').agg(건수=('인증번호', 'size'), 평균갱신횟수=('갱신횟수', 'mean'), 평균유지기간=('유지기간_년', 'mean')).round(2).to_string())
    head('9. 업태 상위 15')
    print(df['업태'].value_counts().head(15).to_string())
    head('10. 시도별 분포 및 평균 갱신횟수')
    print(df.groupby('시도').agg(건수=('인증번호', 'size'), 평균갱신횟수=('갱신횟수', 'mean')).round(2).sort_values('건수', ascending=False).to_string())
    head('11. 다수 인증 보유 업체 상위 10')
    print(df['업체명'].value_counts().head(10).to_string())
    head('12. 만료 임박 업체 (향후 90일 내)')
    soon = df[(df['만료까지_일'] >= 0) & (df['만료까지_일'] <= 90)]
    print(f'90일 내 만료 예정: {len(soon):,}건')
    print(soon.groupby('업종').size().to_string())

def main():
    raw = collect_all()
    df = preprocess(raw)
    os.makedirs('output', exist_ok=True)
    df.to_csv('output/haccp_전처리.csv', index=False, encoding='utf-8-sig')
    eda(df)
    print('\n[완료] 전처리 결과 → output/haccp_전처리.csv')
if __name__ == '__main__':
    main()


[수집] 캐시 사용: data/haccp_인증업체_raw.csv



 1. 기본 구조
행×열: 41,069 × 23
고유 인증번호: 41,069건
고유 업체명  : 24,094개
업체당 평균 인증 수: 1.70건

 2. 결측치
인증번호        0
업체명          0
시도            8
시군구        273
인증일          0
인증종료일      0

 3. ★ 타당성 검증 — 이탈(만료) 업체가 데이터에 존재하는가
인증종료일 < 기준일(2026-09-08) : 0건
인증종료일 >= 기준일                  : 41,069건
인증종료일 최솟값                     : 2026-09-08
→ 만료 건이 0이면 본 API는 '현재 유효 인증'만 담은 스냅샷이며,
  이탈(포기) 업체를 직접 관측할 수 없다.

 4. ★ 갱신주기 복원 가설 검증
갱신주기수별 실제 유지기간(년) — 평균이 3의 배수에 붙고 표준편차가 작으면 가설 성립
             건수    평균  표준편차
갱신주기수                         
0.0             2   1.097     0.378
1.0         11080   3.018     0.160
2.0         11437   5.979     0.273
3.0          7620   8.940     0.300
4.0          4865  11.923     0.344
5.0          2999  14.976     0.641
6.0          2034  17.878     0.587
7.0           553  21.043     0.803
8.0           150  23.529     0.817
9.0            51  26.352     0.507
10.0            8  29.242     0.818

 5. 갱신횟수 분포
갱신횟수
0.0    11082
1.0    11437
2.0     7620
3.0     4865
4.0     

In [3]:
# 03 · 축종·시도별 인증률
import io
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
OUT = 'output'
for _f in ['Malgun Gothic', 'NanumGothic', 'AppleGothic']:
    if any((_f in f.name for f in font_manager.fontManager.ttflist)):
        plt.rcParams['font.family'] = _f
        break
plt.rcParams.update({'axes.unicode_minus': False, 'font.size': 12.5, 'axes.titlesize': 15, 'axes.edgecolor': '#b9c4bd', 'figure.facecolor': 'white', 'xtick.color': '#40514a', 'ytick.color': '#40514a', 'text.color': '#141a17', 'axes.labelcolor': '#40514a'})
C1, C2, C3, C4, MID = ('#c0451c', '#0d7d57', '#ad7e05', '#3f6fb5', '#b9c4bd')
출처 = '국가데이터처 「2026년 2분기 가축동향조사 결과」(2026-07-24 공표, 2026-06-01 기준)'
모집단 = {'한우': (72389, '소이력제 한우 농장수'), '한·육우': (75647, '소이력제 한·육우 농장수'), '젖소': (5342, '소이력제 젖소 농장수'), '돼지': (5465, '돼지이력제 농장수'), '닭': (3123, '3천 마리 이상 가구. 산란계·육용계·종계 합산, 복합은 1가구'), '오리': (502, '2천 마리 이상 가구')}
계절범위 = {'닭': (2747, 3123), '오리': (314, 559)}
통합 = {'전라남도': '전남광주통합특별시', '광주광역시': '전남광주통합특별시'}
시도농장 = pd.DataFrame([('서울특별시', 0, 1, 0, 0, 0), ('부산광역시', 67, 3, 11, 2, 0), ('대구광역시', 685, 17, 48, 15, 0), ('인천광역시', 416, 68, 24, 19, 0), ('광주광역시', 128, 5, 7, 5, 0), ('대전광역시', 146, 0, 1, 0, 0), ('울산광역시', 1397, 18, 21, 10, 0), ('세종특별자치시', 538, 50, 48, 26, 0), ('경기도', 6085, 2077, 1091, 505, 7), ('강원특별자치도', 5362, 243, 236, 137, 2), ('충청북도', 4827, 263, 296, 265, 54), ('충청남도', 9370, 892, 1019, 594, 8), ('전북특별자치도', 8200, 414, 618, 614, 114), ('전라남도', 14011, 377, 522, 358, 269), ('경상북도', 15678, 544, 636, 343, 7), ('경상남도', 8179, 334, 635, 181, 41), ('제주특별자치도', 558, 36, 251, 49, 0)], columns=['시도', '한·육우', '젖소', '돼지', '닭', '오리'])
축종매핑 = {'한우': '한우', '육우': '한·육우', '젖소': '젖소', '돼지': '돼지', '산란계': '닭', '육계': '닭', '오리': '오리'}
농장키 = ['업체명', '시도', '시군구']

def load():
    h = pd.read_csv(OUT + '/haccp_전처리.csv', dtype=str)
    a = h[(h['식품구분'] == '축산물') & (h['업태'] == '가축사육업')].copy()
    for c in 농장키:
        a[c] = a[c].fillna('').str.strip()
    a['축종'] = a['품목'].astype(str).str.replace(' ', '', regex=False).str.split(',')
    복수 = int((a['축종'].str.len() > 1).sum())
    a = a.explode('축종', ignore_index=True)
    print('[로드] 가축사육업 인증 {:,}건 (복수 축종 {:,}건은 축종별로 펼침) / 농장 {:,}곳'.format(a['인증번호'].nunique(), 복수, len(a.drop_duplicates(농장키))))
    return a

def 농장수(d):
    return len(d.drop_duplicates(농장키))

def 축종별(a):
    a = a.copy()
    a['군'] = a['축종'].map(축종매핑)
    a = a.dropna(subset=['군'])
    t = pd.Series({g: 농장수(d) for g, d in a.groupby('군')}, name='인증농가').to_frame()
    한우 = 농장수(a[a['축종'] == '한우'])
    육우 = 농장수(a[a['축종'] == '육우'])
    t.loc['한우', '인증농가'] = 한우
    t.loc['한·육우', '인증농가'] = 농장수(a[a['축종'].isin(['한우', '육우'])])
    t['인증농가'] = t['인증농가'].astype(int)
    한우건 = a[a['축종'] == '한우']
    t['사육농장수'] = [모집단[i][0] for i in t.index]
    t['비고'] = [모집단[i][1] for i in t.index]
    t['인증률%'] = (t['인증농가'] / t['사육농장수'] * 100).round(1)
    t['인증률_하한%'] = [round(t.loc[i, '인증농가'] / 계절범위[i][1] * 100, 1) if i in 계절범위 else np.nan for i in t.index]
    t['인증률_상한%'] = [round(t.loc[i, '인증농가'] / 계절범위[i][0] * 100, 1) if i in 계절범위 else np.nan for i in t.index]
    t = t.sort_values('인증률%', ascending=False)
    print('\n' + '=' * 78)
    print(' 축종별 HACCP 농장 인증률   분모: ' + 출처)
    print('=' * 78)
    print(t[['인증농가', '사육농장수', '인증률%']].to_string())
    print('\n  한우 {:,} / 육우 {:,}  (합계로 한·육우 행을 따로 뒀다)'.format(한우, 육우))
    print('  한우를 세는 방식별: 업체명 {:,}곳 · 업체명+시도+시군구 {:,}곳 · 인증번호 {:,}건'.format(한우건['업체명'].nunique(), 한우, 한우건['인증번호'].nunique()))
    print('    → 인증률 {:.1f}% ~ {:.1f}%. 본문은 업체명+시도+시군구 기준 값을 쓴다.'.format(한우건['업체명'].nunique() / 모집단['한우'][0] * 100, 한우건['인증번호'].nunique() / 모집단['한우'][0] * 100))
    vc = 한우건['업체명'].value_counts()
    print('  한우 인증 {:,}건 중 같은 업체명이 2건 이상인 비율 {:.1f}%'.format(len(한우건), (한우건['업체명'].map(vc) >= 2).mean() * 100))
    print('  닭 분자는 산란계·육계 합집합이다. 둘 다 받은 농장은 한 번만 센다.')
    print('  닭은 산란계·육용계·종계 가구를 합쳐 세는 분모라 축종별로 못 가른다.')
    print('  닭 3천 마리 · 오리 2천 마리 미만 농가는 분모에 없다. 그만큼 값이 높게 나온다.')
    print('\n  [분기 출렁임을 반영한 구간]  분모를 최근 여섯 분기 최소~최대로 바꿔 본 값')
    for i in 계절범위:
        print('    {:<4} {:.1f}% ~ {:.1f}%   (농가 {:,}~{:,})'.format(i, t.loc[i, '인증률_하한%'], t.loc[i, '인증률_상한%'], *계절범위[i]))
    t.to_csv(OUT + '/축종_인증률.csv', encoding='utf-8-sig')
    print('\n[저장] output/축종_인증률.csv')
    return t

def 규모보정(t):
    한육우_전체, 한육우_50미만 = (75647, 54884)
    한육우_50이상 = 한육우_전체 - 한육우_50미만
    한우인증 = int(t.loc['한우', '인증농가'])
    한육우인증 = int(t.loc['한·육우', '인증농가'])
    원래 = t.loc['한우', '인증률%']
    보정 = 한육우인증 / 한육우_50이상 * 100
    최고 = t.drop(index=['한·육우'], errors='ignore')['인증률%'].max()
    print('\n' + '=' * 78)
    print(' 규모 기준을 맞추면 격차는 얼마나 남는가')
    print('=' * 78)
    print('  한·육우 농장 {:,}호 중 50마리 미만이 {:,}호({:.1f}%)다.'.format(한육우_전체, 한육우_50미만, 한육우_50미만 / 한육우_전체 * 100))
    print('  닭·오리 분모에서 소규모를 빼듯, 한우 분모도 50마리 이상으로 좁히면')
    print('    한우 인증률 {:.1f}%  ->  최대 {:.1f}%   (한·육우 인증 {:,} / 50마리 이상 {:,})'.format(원래, 보정, 한육우인증, 한육우_50이상))
    print('  최고 축종(닭 {:.1f}%) 대비 격차 {:.0f}배 -> {:.0f}배.'.format(최고, 최고 / 원래, 최고 / 보정))
    print('\n격차가 줄긴 해도 사라지지는 않는다. 규모만으로는 다 설명되지 않는다는 뜻이다.')
    print("  다만 '24배'를 단독으로 인용하면 안 된다. 그 중 일부는 분모 정의가 만든 값이다.")
    io.open(OUT + '/축종_규모보정.csv', 'w', encoding='utf-8-sig', newline='').write('구분,분모,인증농가,인증률%\n한우 (전체 농장),{},{},{:.1f}\n한·육우 (50마리 이상),{},{},{:.1f}\n'.format(모집단['한우'][0], 한우인증, 원래, 한육우_50이상, 한육우인증, round(보정, 1)))
    print('\n[저장] output/축종_규모보정.csv')
    return 보정

def 시도별(a):
    m = {'한우': '한·육우', '육우': '한·육우', '젖소': '젖소', '돼지': '돼지', '산란계': '닭', '육계': '닭', '오리': '오리'}
    a = a.copy()
    a['군'] = a['축종'].map(m)
    a = a.dropna(subset=['군'])
    시도보정 = {'전라남도': '전남광주통합특별시', '광주광역시': '전남광주통합특별시', '서울시': '서울특별시', '세종시': '세종특별자치시', '세종특별시': '세종특별자치시', '충남': '충청남도', '충북': '충청북도', '경남': '경상남도', '경북': '경상북도'}
    a['시도보정'] = [시도보정.get(str(x).strip(), str(x).strip()) for x in a['시도']]
    a = a.drop_duplicates(['군'] + 농장키)
    분자 = a.groupby(['시도보정', '군']).size().unstack(fill_value=0)
    분모 = 시도농장.copy()
    분모['시도'] = 분모['시도'].map(lambda x: 통합.get(x, x))
    분모 = 분모.groupby('시도').sum()
    공통 = 분모.index.intersection(분자.index)
    빠짐 = sorted(set(분자.index) - set(공통))
    if 빠짐:
        print('\n  [주의] 분모 표에 없는 시도 표기 {}건: {}'.format(len(빠짐), ', '.join(빠짐)))
    rows = []
    for 시도 in 분모.index:
        for 군 in ['한·육우', '젖소', '돼지', '닭', '오리']:
            d = int(분모.loc[시도, 군])
            n = int(분자.loc[시도, 군]) if 시도 in 분자.index and 군 in 분자 else 0
            rows.append({'시도': 시도, '축종군': 군, '인증농가': n, '사육농장수': d, '인증률%': round(n / d * 100, 1) if d else np.nan})
    t = pd.DataFrame(rows)
    t.to_csv(OUT + '/축종_시도.csv', index=False, encoding='utf-8-sig')
    print('\n' + '=' * 78)
    print(' 시도별 인증률(%)  — 사육 농장 30곳 미만인 칸은 비웠다. 한 곳만 받아도 값이 튄다.')
    print('=' * 78)
    p = t.copy()
    p.loc[p['사육농장수'] < 30, '인증률%'] = np.nan
    print(p.pivot(index='시도', columns='축종군', values='인증률%').reindex(columns=['한·육우', '젖소', '돼지', '닭', '오리']).to_string(na_rep='-'))
    넘침 = p[p['인증률%'] > 100]
    if len(넘침):
        print('\n  [100%를 넘는 칸이 있다 — 숨기지 않고 남긴다]')
        for _, r in 넘침.iterrows():
            print('    {} {} {:,}/{:,} = {:.1f}%'.format(r['시도'], r['축종군'], int(r['인증농가']), int(r['사육농장수']), r['인증률%']))
        print('    이건 오류가 아니라 위 한계 1·2가 눈에 보이는 자리다.')
        print('    산란계와 육계를 각각 받은 한 농가가 분자에서 두 번 세어지고,')
        print('    3천 마리 미만 농가는 분모에서 빠져 있다. 닭은 순위만 읽어야 한다.')
    print('\n[저장] output/축종_시도.csv')
    h = t[(t['축종군'] == '한·육우') & (t['사육농장수'] >= 500)].sort_values('사육농장수', ascending=False).rename(columns={'인증농가': '인증', '사육농장수': '농장', '인증률%': '인증률'})[['시도', '인증', '농장', '인증률']]
    h.to_csv(OUT + '/한육우_시도별_인증률.csv', index=False, encoding='utf-8-sig')
    print('\n[한·육우 시도별]  기획서 [그림 5]')
    print(h.to_string(index=False))
    print('[저장] output/한육우_시도별_인증률.csv')
    return t

def 그림(t):
    d = t.drop(index=['한·육우'], errors='ignore').sort_values('인증률%')
    f, ax = plt.subplots(figsize=(11.2, 5.6), dpi=150)
    색 = [C1 if v < 5 else C3 if v < 25 else C2 for v in d['인증률%']]
    ax.barh(d.index, d['인증률%'], color=색, height=0.62)
    for i, (nm, r) in enumerate(d.iterrows()):
        ax.text(r['인증률%'] + 1.2, i, '{:.1f}%  ({:,}/{:,})'.format(r['인증률%'], int(r['인증농가']), int(r['사육농장수'])), va='center', fontsize=11.5, color='#141a17', fontweight='bold')
    ax.set_xlim(0, max(d['인증률%']) * 1.42)
    ax.set_xlabel('HACCP 인증 농가 비율 (%)')
    ax.spines[['top', 'right']].set_visible(False)
    배수 = d['인증률%'].max() / d['인증률%'].min()
    ax.set_title('축종에 따라 농장 HACCP은 {:.0f}배 갈린다'.format(배수), loc='left', fontsize=15.5, fontweight='bold', pad=52)
    ax.text(0, 1.035, '분모는 국가데이터처 가축동향조사 2026년 2분기(2026-06-01 기준) 사육 농장(가구)수.\n닭은 산란계·육용계 합산 상한값. 닭 3천 마리·오리 2천 마리 미만 농가는 분모에 없다.', transform=ax.transAxes, fontsize=11, color='#40514a', va='bottom', linespacing=1.5)
    f.tight_layout()
    f.savefig(OUT + '/그림39_축종인증률.png', bbox_inches='tight', facecolor='white')
    plt.close(f)
    print('[저장] output/그림39_축종인증률.png')
if __name__ == '__main__':
    a = load()
    t = 축종별(a)
    규모보정(t)
    시도별(a)
    그림(t)
    낮 = t.drop(index=['한·육우'], errors='ignore')['인증률%']
    print('\n' + '=' * 78)
    print(' 읽는 법')
    print('=' * 78)
    print(' · 최저 {:.1f}% ~ 최고 {:.1f}%. 축종 사이가 {:.0f}배 갈린다.'.format(낮.min(), 낮.max(), 낮.max() / 낮.min()))
    print(' · 한우는 농가가 7만 곳이 넘는데 인증은 {:.1f}%다. 남은 여지가 가장 큰 축종이다.'.format(t.loc['한우', '인증률%']))
    print(' · 다만 한우 농가 대부분은 소규모다. 식품 쪽 결론과 같은 함정이 있다.')
    print("   '남은 수가 많다'가 곧 '받을 수 있다'는 아니다. 규모별 분해가 다음 과제다.")


[로드] 가축사육업 인증 5,972건 (복수 축종 92건은 축종별로 펼침) / 농장 5,834곳

 축종별 HACCP 농장 인증률   분모: 국가데이터처 「2026년 2분기 가축동향조사 결과」(2026-07-24 공표, 2026-06-01 기준)
         인증농가  사육농장수  인증률%
닭           1918        3123     61.4
돼지         1322        5465     24.2
오리          105         502     20.9
젖소          629        5342     11.8
한우         1835       72389      2.5
한·육우      1845       75647      2.4

  한우 1,835 / 육우 73  (합계로 한·육우 행을 따로 뒀다)
  한우를 세는 방식별: 업체명 1,676곳 · 업체명+시도+시군구 1,835곳 · 인증번호 1,884건
    → 인증률 2.3% ~ 2.6%. 본문은 업체명+시도+시군구 기준 값을 쓴다.
  한우 인증 1,884건 중 같은 업체명이 2건 이상인 비율 17.8%
  닭 분자는 산란계·육계 합집합이다. 둘 다 받은 농장은 한 번만 센다.
  닭은 산란계·육용계·종계 가구를 합쳐 세는 분모라 축종별로 못 가른다.
  닭 3천 마리 · 오리 2천 마리 미만 농가는 분모에 없다. 그만큼 값이 높게 나온다.

  [분기 출렁임을 반영한 구간]  분모를 최근 여섯 분기 최소~최대로 바꿔 본 값
    닭    61.4% ~ 69.8%   (농가 2,747~3,123)
    오리   18.8% ~ 33.4%   (농가 314~559)

[저장] output/축종_인증률.csv

 규모 기준을 맞추면 격차는 얼마나 남는가
  한·육우 농장 75,647호 중 50마리 미만이 54,884호(72.6%)다.
  닭·오리 분모에서 소규모를 빼듯, 한우 분모도 50마리 이상으로 좁히면
    한우 인증률 2.5%  ->  최대 

[저장] output/그림39_축종인증률.png

 읽는 법
 · 최저 2.5% ~ 최고 61.4%. 축종 사이가 25배 갈린다.
 · 한우는 농가가 7만 곳이 넘는데 인증은 2.5%다. 남은 여지가 가장 큰 축종이다.
 · 다만 한우 농가 대부분은 소규모다. 식품 쪽 결론과 같은 함정이 있다.
   '남은 수가 많다'가 곧 '받을 수 있다'는 아니다. 규모별 분해가 다음 과제다.


In [4]:
# 04 · 신규 인증 추이
import io
import os
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
OUT = 'output'
FIG = os.path.join(OUT, '기획서_그림')
농장키 = ['업체명', '시도', '시군구']
축종군 = {'한우': '한우', '육우': '한우', '젖소': '젖소', '돼지': '돼지', '산란계': '닭', '육계': '닭', '오리': '오리'}
순서 = ['닭', '한우', '돼지', '젖소', '오리']

def load():
    h = pd.read_csv(OUT + '/haccp_전처리.csv', dtype=str)
    a = h[(h['식품구분'] == '축산물') & (h['업태'] == '가축사육업')].copy()
    for c in 농장키:
        a[c] = a[c].fillna('').str.strip()
    a['축종'] = a['품목'].astype(str).str.replace(' ', '', regex=False).str.split(',')
    a = a.explode('축종', ignore_index=True)
    a['인증일'] = pd.to_datetime(a['인증일'], errors='coerce')
    a['군'] = a['축종'].map(축종군)
    a = a.dropna(subset=['군', '인증일'])
    f = a.sort_values('인증일').drop_duplicates(['군'] + 농장키).copy()
    f['연'] = f['인증일'].dt.year
    print('[로드] 가축사육업 농장 {:,}곳 (축종군별)'.format(len(f)))
    return f

def 묶음(f):
    rows = []
    for g in 순서:
        s = f[f['군'] == g]
        c = s.groupby('인증일').size()
        rows.append({'축종': g, '인증농가': len(s), '고유인증일': len(c), '최대동일자': int(c.max()), '동일자2곳이상%': round(c[c >= 2].sum() / len(s) * 100, 1), '동일자5곳이상%': round(c[c >= 5].sum() / len(s) * 100, 1)})
    t = pd.DataFrame(rows)
    print()
    print('=' * 78)
    print(' 묶음 신청의 흔적 — 같은 날 몇 곳이 함께 들어왔는가')
    print('=' * 78)
    print(t.to_string(index=False))
    t.to_csv(OUT + '/축종_동일자묶음.csv', index=False, encoding='utf-8-sig')
    닭 = t[t['축종'] == '닭'].iloc[0]
    한우 = t[t['축종'] == '한우'].iloc[0]
    print()
    print(' 같은 날 5곳 이상: 한우 {:.1f}%  vs  닭 {:.1f}%'.format(한우['동일자5곳이상%'], 닭['동일자5곳이상%']))
    print(' 한우가 더 뭉쳐 있다. **계열화 묶음 가설은 지지되지 않는다.**')
    print(' 두 축종 모두 동일자 묶음이 있는데, 이건 계열화보다 심사 일정의 흔적으로 보인다.')
    return t

def 추세(f):
    t = f.pivot_table(index='연', columns='군', values='업체명', aggfunc='count').fillna(0).astype(int).reindex(columns=순서).loc[2015:2025]
    print()
    print('=' * 78)
    print(' 연도별 신규 인증 농가')
    print('=' * 78)
    print(t.to_string())
    t.to_csv(OUT + '/축종_연도별신규.csv', encoding='utf-8-sig')
    최근 = t.loc[2021:2025].sum()
    print()
    print(' [최근 5년 합계]')
    for g in 순서:
        print('   {:<4} {:>5,}곳  (연 {:.1f}곳)'.format(g, 최근[g], 최근[g] / 5))
    한우 = t.loc[2021:2025, '한우']
    print()
    print(' [기획서 1-1] 한우·육우 신규 인증 2021~2025년 {}곳'.format('·'.join((str(v) for v in 한우))))
    print('   연평균 {:.1f}곳 = 한우 농장 72,389곳의 연 {:.2f}%'.format(한우.mean(), 한우.mean() / 72389 * 100))
    print()
    print(' 최근 5년만 보면 한우가 닭보다 많다. 한우가 느린 게 아니다.')
    return t

def 부문별():
    h = pd.read_csv(OUT + '/haccp_전처리.csv', dtype=str)
    h['인증일'] = pd.to_datetime(h['인증일'], errors='coerce')
    h = h.dropna(subset=['인증일'])
    h['연'] = h['인증일'].dt.year
    h['부문'] = '축산물 가공·유통'
    h.loc[h['식품구분'] == '식품', '부문'] = '식품'
    h.loc[(h['식품구분'] == '축산물') & (h['업태'] == '가축사육업'), '부문'] = '축산물 농장'
    for c in 농장키:
        h[c] = h[c].fillna('').str.strip()
    f = h.sort_values('인증일').drop_duplicates(['부문'] + 농장키)
    t = f.pivot_table(index='연', columns='부문', values='업체명', aggfunc='count').fillna(0).astype(int).reindex(columns=['식품', '축산물 가공·유통', '축산물 농장'])
    t['합계'] = t.sum(axis=1)
    t = t.loc[2015:2025]
    print()
    print('=' * 78)
    print(' 부문별 신규 인증 업소 — 인증원 전체가 막혔나, 농장만 막혔나')
    print('=' * 78)
    print(t.to_string())
    t.to_csv(OUT + '/부문별_연도별신규.csv', encoding='utf-8-sig')
    print()
    print(' 합계 {:,} -> {:,}  (처리 역량은 두 배 넘게 커졌다)'.format(t.loc[2015, '합계'], t.loc[2025, '합계']))
    print(' 농장 {:,} -> {:,}  (농장만 줄었다)'.format(t.loc[2015, '축산물 농장'], t.loc[2025, '축산물 농장']))
    print(' 2019년 이후 농장 연 {:,}~{:,}개'.format(t.loc[2019:, '축산물 농장'].min(), t.loc[2019:, '축산물 농장'].max()))
    return t

def 그림_신규추이(t):
    for _f in ['Malgun Gothic', 'NanumGothic', 'AppleGothic']:
        if any((_f in x.name for x in font_manager.fontManager.ttflist)):
            plt.rcParams['font.family'] = _f
            break
    plt.rcParams['axes.unicode_minus'] = False
    색 = {'식품': '#a9b1ad', '축산물 가공·유통': '#5e6863', '축산물 농장': '#b8411a'}
    글자, 격자 = ('#4c5752', '#dee4e0')
    fig, ax = plt.subplots(figsize=(14.82, 7.22), dpi=100)
    x = list(t.index)
    for 부문 in ['식품', '축산물 가공·유통', '축산물 농장']:
        농장 = 부문 == '축산물 농장'
        ax.plot(x, t[부문], color=색[부문], lw=5 if 농장 else 3.2, marker='o', ms=10 if 농장 else 9, zorder=3 if 농장 else 2)
    끝 = x[-1]
    가공, 식품 = (t.loc[끝, '축산물 가공·유통'], t.loc[끝, '식품'])
    위, 아래 = (max(가공, 식품), min(가공, 식품))
    if 위 - 아래 < 110:
        위, 아래 = ((위 + 아래) / 2 + 55, (위 + 아래) / 2 - 55)
    ax.text(끝 + 0.25, 위 if 가공 >= 식품 else 아래, '축산물 가공·유통', color=색['축산물 가공·유통'], fontsize=19, va='center')
    ax.text(끝 + 0.25, 아래 if 가공 >= 식품 else 위, '식품', color=색['축산물 가공·유통'], fontsize=19, va='center')
    ax.text(끝 + 0.25, t.loc[끝, '축산물 농장'], '축산물 농장', color=색['축산물 농장'], fontsize=19, va='center', fontweight='bold')
    for 연 in (x[0], 끝):
        v = int(t.loc[연, '축산물 농장'])
        ax.text(연 + (0.3 if 연 == x[0] else -0.2), v + 70, '{:,}'.format(v), color=색['축산물 농장'], fontsize=17, ha='center', va='bottom')
    ax.set_xlim(x[0] - 0.4, 끝 + 2.3)
    ax.set_xticks(x[::2])
    ax.set_ylim(100, ax.get_ylim()[1])
    ax.grid(axis='y', color=격자, lw=1.2)
    ax.set_axisbelow(True)
    ax.tick_params(colors=글자, labelsize=18, length=0, pad=10)
    for sp in ['top', 'right', 'left']:
        ax.spines[sp].set_visible(False)
    ax.spines['bottom'].set_color(격자)
    fig.subplots_adjust(left=0.07, right=0.97, top=0.97, bottom=0.1)
    os.makedirs(FIG, exist_ok=True)
    fig.savefig(os.path.join(FIG, '그림_신규추이.png'), facecolor='white')
    plt.close(fig)
    print('[저장] output/기획서_그림/그림_신규추이.png  (기획서 [그림 1])')
if __name__ == '__main__':
    f = load()
    묶음(f)
    추세(f)
    누적 = f.groupby('군').size().reindex(순서)
    print()
    print('=' * 78)
    print(' 결론')
    print('=' * 78)
    print(' 인증받은 농가 수:  ' + ' · '.join(('{} {:,}'.format(g, 누적[g]) for g in 순서)))
    print(' 분자는 축종 사이에 큰 차이가 없다. 차이는 대부분 분모(산업 구조)에 있다.')
    print(' 닭은 전국 농가가 3,123곳뿐이고 한우는 72,389곳이다. 23배다.')
    print()
    print(' 한우의 {:.1f}%는 인증을 덜 받아서가 아니다.'.format(누적['한우'] / 72389 * 100))
    print(' 비슷한 수를 받았는데 나눌 모집단이 23배 크기 때문이다.')
    print(" 그래서 '농가를 더 설득하자'가 아니라 '단위를 바꾸자'가 답이 된다.")
    그림_신규추이(부문별())


[로드] 가축사육업 농장 5,819곳 (축종군별)

 묶음 신청의 흔적 — 같은 날 몇 곳이 함께 들어왔는가
축종  인증농가  고유인증일  최대동일자  동일자2곳이상%  동일자5곳이상%
  닭      1918        1314           8            52.5             4.5
한우      1845        1090          16            62.9            15.6
돼지      1322         970           5            45.8             1.1
젖소       629         491           9            37.2             5.2
오리       105          97           3            14.3             0.0

 같은 날 5곳 이상: 한우 15.6%  vs  닭 4.5%
 한우가 더 뭉쳐 있다. **계열화 묶음 가설은 지지되지 않는다.**
 두 축종 모두 동일자 묶음이 있는데, 이건 계열화보다 심사 일정의 흔적으로 보인다.

 연도별 신규 인증 농가
군     닭  한우  돼지  젖소  오리
연                               
2015  184   164    94    43    18
2016  151   118    91    38    20
2017   93   104    79    13     9
2018  158   103    55    28    13
2019  123    60    55    35     3
2020   99    72    44    48     4
2021   78   116    64    45     0
2022  101   100    36    62     3
2023   72   115    59    47     0
2024   53   101    33    32     2
2025   80    92 


 부문별 신규 인증 업소 — 인증원 전체가 막혔나, 농장만 막혔나
부문  식품  축산물 가공·유통  축산물 농장  합계
연                                             
2015   323               272          505  1100
2016   440               296          418  1154
2017   589               220          306  1115
2018   596               328          361  1285
2019   732               425          276  1433
2020  1121              1070          269  2460
2021  1388              1686          308  3382
2022  1081               823          303  2207
2023   749               832          294  1875
2024   997              1247          226  2470
2025   961              1126          258  2345

 합계 1,100 -> 2,345  (처리 역량은 두 배 넘게 커졌다)
 농장 505 -> 258  (농장만 줄었다)
 2019년 이후 농장 연 226~308개
[저장] output/기획서_그림/그림_신규추이.png  (기획서 [그림 1])


In [5]:
# 05 · 스마트HACCP 중요관리점
import io
import os
import sys
import time
import requests
import pandas as pd
OUT = 'output'
SERVICE_KEY = os.environ.get('HACCP_API_KEY', '여기에_공공데이터포털_인증키')
API = {'축산물': 'https://apis.data.go.kr/B553748/SmartCertLiveStockListService/getLiveStockList', '기술지원': 'https://apis.data.go.kr/B553748/CounselSmartListService/getCounselSmartListService'}
PAGE = 100

def 받기(이름):
    url = API[이름]
    rows, page, total = ([], 1, None)
    while True:
        for 시도 in range(5):
            try:
                r = requests.get('{}?serviceKey={}&pageNo={}&numOfRows={}&returnType=json'.format(url, SERVICE_KEY, page, PAGE), timeout=120)
                j = r.json()
                break
            except Exception as e:
                print('   재시도 {} ({})'.format(시도 + 1, str(e)[:60]), flush=True)
                time.sleep(3)
        else:
            sys.exit('[중단] {} {}페이지를 받지 못했다'.format(이름, page))
        body = j['body']
        total = int(body['totalCount'])
        items = body.get('items') or []
        if isinstance(items, dict):
            items = items.get('item', [])
            items = items if isinstance(items, list) else [items]
        for it in items:
            rows.append(it.get('item', it))
        print('[{}] {}페이지 누적 {:,} / {:,}'.format(이름, page, len(rows), total), flush=True)
        if len(rows) >= total or not items:
            break
        page += 1
    d = pd.DataFrame(rows)
    p = 'data/smart_{}.csv'.format(이름)
    d.to_csv(p, index=False, encoding='utf-8-sig')
    print('[저장] {}  {:,}건'.format(p, len(d)))
    return d

def 분석(sm, cs):
    print()
    print('=' * 78)
    print(' ① 축산물 스마트HACCP 등록 업체 — 업태별')
    print('=' * 78)
    sm = sm.drop_duplicates('appointno')
    t = sm['businesstype'].value_counts().rename('스마트등록').to_frame()
    h = pd.read_csv('data/haccp_인증업체_raw.csv', dtype=str)
    h = h[h['productGb'] == '축산물'].drop_duplicates('appointno')
    base = h['businesstypeNm'].value_counts()
    t['HACCP인증'] = t.index.map(base).fillna(0).astype(int)
    for 업태 in ['가축사육업']:
        if 업태 not in t.index:
            t.loc[업태] = [0, int(base.get(업태, 0))]
    t['스마트전환%'] = (t['스마트등록'] / t['HACCP인증'].where(t['HACCP인증'] > 0) * 100).round(2)
    t = t.sort_values('스마트등록', ascending=False)
    print(t.to_string())
    t.to_csv(OUT + '/스마트_업태별.csv', encoding='utf-8-sig')
    농장 = int(t.loc['가축사육업', '스마트등록']) if '가축사육업' in t.index else 0
    농장인증 = int(base.get('가축사육업', 0))
    print()
    print(' 가축사육업(농장): HACCP 인증 {:,}건 중 스마트HACCP 등록 {:,}건'.format(농장인증, 농장))
    print()
    print('=' * 78)
    print(' ② 스마트HACCP 기술지원 요청 업체 — 업종별 (등록 전 파이프라인)')
    print('=' * 78)
    if len(cs):
        c = cs['indutyNm'].fillna('미상').value_counts()
        print(c.head(25).to_string())
        가공 = c[c.index.str.contains('식육|축산물|유가공|알가공|식용란', na=False) & (c.index != '가축사육업')]
        print()
        print(' 축산물 가공·유통 {:,}건 / 농장 {:,}건 / 전체 {:,}건'.format(int(가공.sum()), int(c.get('가축사육업', 0)), len(cs)))
        c.to_csv(OUT + '/스마트_기술지원_업종.csv', encoding='utf-8-sig')

def 중요관리점(sm):
    sm = sm.drop_duplicates('appointno')
    c = sm['ccp'].dropna().str.split(',').explode().str.strip()
    c = c[c != '']
    규칙 = [('금속', '금속검출'), ('냉장', '냉장·냉동 보관'), ('냉동', '냉장·냉동 보관'), ('동결', '냉장·냉동 보관'), ('살균', '가열·살균'), ('가열', '가열·살균'), ('열처리', '가열·살균'), ('자숙', '가열·살균'), ('훈연', '가열·살균'), ('이물', '이물 검출'), ('X-ray', '이물 검출'), ('엑스레이', '이물 검출')]

    def 군(x):
        for k, v in 규칙:
            if k in x:
                return v
        return '기타'
    g = c.map(군).value_counts()
    기계 = g.drop(['기타'], errors='ignore').sum()
    print()
    print('=' * 78)
    print(' ③ 스마트HACCP 등록 업체의 중요관리점 — 무엇을 자동 기록하는가')
    print('=' * 78)
    print(' 등록 {:,}건 / 중요관리점 표기 {:,}개'.format(len(sm), len(c)))
    print(g.to_string())
    print(' 온도·검출기·가열처럼 기계가 기록하는 항목 {:.1f}%'.format(기계 / len(c) * 100))
    pd.DataFrame([{'등록': len(sm), 'CCP표기': len(c), '냉장냉동': int(g.get('냉장·냉동 보관', 0)), '금속검출': int(g.get('금속검출', 0)), '가열살균': int(g.get('가열·살균', 0)), '이물검출': int(g.get('이물 검출', 0)), '기타': int(g.get('기타', 0)), '기계기록형%': round(기계 / len(c) * 100, 1)}]).to_csv(OUT + '/스마트_CCP유형.csv', index=False, encoding='utf-8-sig')
    print('[저장] output/스마트_CCP유형.csv')
if __name__ == '__main__':
    mode = 'analyze' if all((os.path.exists(p) for p in ['data/smart_축산물.csv', 'data/smart_기술지원.csv'])) else 'collect'
    if mode in ('all', 'collect'):
        sm = 받기('축산물')
        cs = 받기('기술지원')
    else:
        sm = pd.read_csv('data/smart_축산물.csv', dtype=str)
        cs = pd.read_csv('data/smart_기술지원.csv', dtype=str)
    분석(sm, cs)
    중요관리점(sm)



 ① 축산물 스마트HACCP 등록 업체 — 업태별


                               스마트등록  HACCP인증  스마트전환%
businesstype                                                     
식육포장처리업                        150       4567         3.28
축산물가공업-식육가공업                79       3197         2.47
축산물가공업-유가공업                  27        395         6.84
축산물판매업-식육판매업                 9        549         1.64
식육가공업                              9          0          NaN
축산물가공업-알가공업                   6        228         2.63
식용란선별포장업                        6        589         1.02
축산물판매업-식용란수집판매업           5       1009         0.50
유가공업                                3          0          NaN
알가공업                                3          0          NaN
식육즉석판매가공업                      1         38         2.63
가축사육업                              0       5972         0.00

 가축사육업(농장): HACCP 인증 5,972건 중 스마트HACCP 등록 0건

 ② 스마트HACCP 기술지원 요청 업체 — 업종별 (등록 전 파이프라인)
indutyNm
식품제조가공업                       1516
식육포장처리업                        585
축산물가공업-식육가공업      

In [6]:
# 06 · 컨설팅 예산·확산 지표
import pandas as pd
OUT = 'output'
컨설팅_예산 = {2019: 2800, 2020: 2800, 2021: 2593, 2022: 2593, 2023: 2033, 2024: 1015}
단가_한우 = 6.0
단가_돼지 = 8.0
단가_단체_농가 = 3.0
한육우_50이상 = 20763
로드맵 = [(1, 2024, '집유장', '집유장', '', '', '2개소 · 187.5백만원'), (2, 2025, '도축장(소·돼지), 농장(젖소)', '도축장(포유류), 집유장', '젖소', '', '국비 551백만원 · 업체당 국비 40백만원'), (3, 2026, '도축장(가금), 농장(소·돼지·가금)', '가금류 도축장, 도축장·집유장 8개소', '소·돼지·가금', '', '구축비 1억 이내 최대 70%')]

def 부문별():
    h = pd.read_csv('data/haccp_인증업체_raw.csv', dtype=str)
    a = h[h['productGb'] == '축산물'].drop_duplicates('appointno')
    sm = pd.read_csv('data/smart_축산물.csv', dtype=str).drop_duplicates('appointno')
    s = sm['businesstype'].value_counts()
    t = a.groupby(['companykindNm', 'businesstypeNm']).size().rename('HACCP').to_frame()
    t['스마트'] = [int(s.get(b, 0)) for _, b in t.index]
    g = t.groupby(level=0).sum()
    cs = pd.read_csv('data/smart_기술지원.csv', dtype=str)
    f = cs['indutyNm'].fillna('')
    요청 = {'제조,가공': int(f.str.contains('식육포장|식육가공|유가공|알가공').sum()), '유통,판매,조리': int(f.str.contains('식육판매|식용란|축산물유통전문|식육즉석|축산물운반|축산물보관').sum()), '농장': int((f == '가축사육업').sum()) + int(f.str.contains('종축|부화').sum()), '사료': int(f.str.contains('사료').sum())}
    g['기술지원요청'] = [요청.get(i, 0) for i in g.index]
    g['스마트전환%'] = (g['스마트'] / g['HACCP'] * 100).round(2)
    g['요청률%'] = (g['기술지원요청'] / g['HACCP'] * 100).round(2)
    g = g.reindex(['제조,가공', '유통,판매,조리', '사료', '농장'])
    g.index = ['축산물 가공', '축산물 유통·판매', '사료', '농장']
    g.to_csv(OUT + '/축산스마트_부문별.csv', encoding='utf-8-sig')
    print('[부문별]')
    print(g.to_string())
    return (g, len(cs))

def 예산():
    bu = pd.read_csv(OUT + '/부문별_연도별신규.csv', index_col=0)
    rows = []
    for y, b in 컨설팅_예산.items():
        rows.append({'연도': y, '컨설팅예산_백만원': b, '컨설팅가능농장_상한_한우단가': int(b // 단가_한우), '컨설팅가능농장_상한_돼지단가': int(b // 단가_돼지), '농장신규인증': int(bu.loc[y, '축산물 농장'])})
    t = pd.DataFrame(rows)
    t.to_csv(OUT + '/축산스마트_예산.csv', index=False, encoding='utf-8-sig')
    print('\n[컨설팅 예산과 농장 신규 인증]')
    print(t.to_string(index=False))
    감소 = (1 - 컨설팅_예산[2024] / 컨설팅_예산[2019]) * 100
    print('  2019 -> 2024 예산 {:.0f}% 감소'.format(감소))
    개별 = int(컨설팅_예산[2024] // 단가_한우)
    단체 = int(컨설팅_예산[2024] // 단가_단체_농가)
    print('  [기획서] 2024년 예산 전액을 한우에 쓴다고 가정하면')
    print('    개별 컨설팅(농장당 600만 원) {}곳 · 단체 컨설팅(농가당 300만 원, 기본료 제외) 최대 {}곳'.format(개별, 단체))
    print('    50마리 이상 한·육우 {:,}곳 ÷ 연 {}곳 = {:.1f}년 이상'.format(한육우_50이상, 단체, 한육우_50이상 / 단체))
    print('  주의: 예산에는 도축장·집유장·사료공장 컨설팅, 방역 컨설팅, 살모넬라 백신이 함께 들어 있다.')
    print("        그래서 '컨설팅 가능 농장'은 예산 전부를 농장에 썼을 때의 상한이다.")
    return (t, 감소)
if __name__ == '__main__':
    g, 요청전체 = 부문별()
    t, 감소 = 예산()
    pd.DataFrame(로드맵, columns=['연차', '연도', '계획', '실행', '농장_계획', '농장_실행', '규모']).to_csv(OUT + '/축산스마트_로드맵.csv', index=False, encoding='utf-8-sig')
    print('\n[로드맵 계획 대비 실행]  계획 연차에 농장이 든 해: 2025(젖소), 2026(소·돼지·가금) / 실행: 0')
    가공유통 = g.loc[['축산물 가공', '축산물 유통·판매']]
    벤치율 = 가공유통['기술지원요청'].sum() / 가공유통['HACCP'].sum() * 100
    농장 = g.loc['농장']
    지표 = pd.DataFrame([{'농장_HACCP': int(농장['HACCP']), '농장_스마트': int(농장['스마트']), '농장_기술지원요청': int(농장['기술지원요청']), '기술지원요청_전체': 요청전체, '가공유통_요청률%': round(벤치율, 2), '농장_요청률%': round(농장['기술지원요청'] / 농장['HACCP'] * 100, 3), '농장_벤치마크요청수': int(round(농장['HACCP'] * 벤치율 / 100)), '컨설팅예산_감소%': round(감소, 0), '2024_컨설팅가능농장_상한': '{}~{}'.format(int(컨설팅_예산[2024] // 단가_돼지), int(컨설팅_예산[2024] // 단가_한우)), '2024_한우컨설팅_개별~단체': '{}~{}'.format(int(컨설팅_예산[2024] // 단가_한우), int(컨설팅_예산[2024] // 단가_단체_농가)), '50마리이상_도달연수': round(한육우_50이상 / (컨설팅_예산[2024] // 단가_단체_농가), 1), '2024_농장신규': int(t.loc[t['연도'] == 2024, '농장신규인증'].iloc[0])}])
    지표.to_csv(OUT + '/축산스마트_지표.csv', index=False, encoding='utf-8-sig')
    print('\n[핵심 지표]')
    print(지표.T.to_string(header=False))


[부문별]
                  HACCP  스마트  기술지원요청  스마트전환%  요청률%
축산물 가공        8387     262          1062         3.12    12.66
축산물 유통·판매   2425      21            92         0.87     3.79
사료                270       0             0         0.00     0.00
농장               6294       0             1         0.00     0.02

[컨설팅 예산과 농장 신규 인증]
 연도  컨설팅예산_백만원  컨설팅가능농장_상한_한우단가  컨설팅가능농장_상한_돼지단가  농장신규인증
 2019               2800                           466                           350           276
 2020               2800                           466                           350           269
 2021               2593                           432                           324           308
 2022               2593                           432                           324           303
 2023               2033                           338                           254           294
 2024               1015                           169                           126           226
  2019 -> 2024 예